# Processing Cricket Spectrograms

Pulls four numbers out of every SINA cricket spectrogram: element length,
inter-element interval, inter-burst interval, and elements per burst.
Hapithus melodius is skipped--its chirps speed up over the course of a
call, which doesn't fit this four-number model. See the README for what each
column means.

Stores the result as `cricket_final` for `Display_Function.ipynb`.

In [ ]:
import re
import statistics
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image, ImageOps

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

# Root directory: one subfolder per species, each holding that species' spectrogram files.
CRICKETS_DIR = Path.home() / 'Discrete_Signals' / 'Crickets'

## Constants

In [ ]:
# x-axis time span supplied by hand (OCR unreliable for these files).
MANUAL_MAX_SECONDS = {
    "Allonemobius_shalontaki_spectrogram_548ss.gif":      0.7,
    "Oecanthus_rileyi_spectrogram_588salb.jpg":           1.0,
    "Oecanthus_salvii_spectrogram_573sl2speca.jpg":       1.0,
    "Oecanthus_alexanderi_spectrogram_575ss.gif":         5.0,
    "Oecanthus_laricis_spectrogram_591ss2.gif":           1.0,
    "Anurogryllus_arboreus_spectrogram_491ss2.gif":       1.0,
    "Anurogryllus_celerinictus_spectrogram_492ss2.gif":   1.0,
}

# Resolution too low for automatic detection--values returned as-is.
# The per-entry comment says what the pipeline gets wrong.
MANUAL_RESULTS = {
    "Neonemobius_palustris_spectrogram_524ss2.gif": {
        "element_length":         0.0083,
        "inter_element_interval": 0.0083,
        "inter_burst_interval":   0.0,
        "elements_per_burst":     1,
    },
    # Pairs are visible in the right half of the image; left-half IEI gaps
    # are below pixel resolution and merge into single-element detections.
    "Cycloptilum_velox_spectrogram_452ss2.gif": {
        "element_length":         0.0242,
        "inter_element_interval": 0.0043,
        "inter_burst_interval":   0.1433,
        "elements_per_burst":     2,
    },
    # Continuous trill; gap ratio of 4.03 barely trips the burst threshold
    # due to slight rhythm variation--biologically EPB = 1.
    "Anurogryllus_celerinictus_spectrogram_492ss2.gif": {
        "element_length":         0.007,
        "inter_element_interval": 0.006,
        "inter_burst_interval":   0.0,
        "elements_per_burst":     1,
    },
    # 3-cluster gap structure (tiny IEI / medium artifact / large IBI).
    # Algorithm merges tiny+medium as IEI and returns EPB=3; correct is 2.
    "Cycloptilum_comprehendens_spectrogram_437ss2.gif": {
        "element_length":         0.0192,
        "inter_element_interval": 0.0064,
        "inter_burst_interval":   0.3958,
        "elements_per_burst":     2,
    },
    # IEI ~1.3 px at this resolution; narrow gaps occasionally missed, merging
    # 4-element bursts into apparent 2s--algorithm gives median=2 at all thresholds.
    "Gryllus_makhosica_spectrogram_726ss2.jpg": {
        "element_length":         0.0578,
        "inter_element_interval": 0.0191,
        "inter_burst_interval":   0.3539,
        "elements_per_burst":     4,
    },
    # Faint first element undetectable at any threshold; EPB=3 at all thresholds.
    "Gryllus_brevicaudus_spectrogram_465ss3wg.jpg": {
        "element_length":         0.0183,
        "inter_element_interval": 0.0223,
        "inter_burst_interval":   0.3240,
        "elements_per_burst":     4,
    },
    # Faint first element; algorithm returns mixed 2-4 list, median=3.
    "Gryllus_montis_spectrogram_727ss2.jpg": {
        "element_length":         0.0388,
        "inter_element_interval": 0.0148,
        "inter_burst_interval":   0.3983,
        "elements_per_burst":     4,
    },
    # Faint first element in a noisy spectrogram; EPB=3 at all thresholds.
    "Gryllus_saxatilis_spectrogram_731ss2.jpg": {
        "element_length":         0.0331,
        "inter_element_interval": 0.0165,
        "inter_burst_interval":   0.3491,
        "elements_per_burst":     4,
    },
    # Faint first element; algorithm sees groups of 6-7, visual is 7.
    "Gryllus_thinos_spectrogram_734ss3.jpg": {
        "element_length":         0.0165,
        "inter_element_interval": 0.0372,
        "inter_burst_interval":   1.1016,
        "elements_per_burst":     7,
    },
    # Low resolution; faint elements cause EPB to bounce 3-7 across thresholds.
    "Allonemobius_socius_spectrogram_532ss2.gif": {
        "element_length":         0.0115,
        "inter_element_interval": 0.0044,
        "inter_burst_interval":   0.2273,
        "elements_per_burst":     8,
    },
    # Algorithm detects false burst structure (EPB=4, IBI=0.085s); visually a trill.
    "Oecanthus_exclamationis_spectrogram_590ss2.gif": {
        "element_length":         0.0059,
        "inter_element_interval": 0.0152,
        "inter_burst_interval":   0.0,
        "elements_per_burst":     1,
    },
    # Algorithm detects false burst structure (EPB=3, IBI=0.324s); visually a trill.
    "Neoxabea_bipunctata_spectrogram_601ss2.gif": {
        "element_length":         0.0305,
        "inter_element_interval": 0.0925,
        "inter_burst_interval":   0.0,
        "elements_per_burst":     1,
    },
}

# Force a specific detection threshold for files whose faint first element
# only shows up below the scorer's usual pick.
FAINT_ELEMENT_THRESHOLD = {
    "Gryllodes_sigillatus_spectrogram_501ss2.gif":  0.38,
    "Gryllus_chisosensis_spectrogram_721ss2.jpg":   0.30,
    "Gryllus_longicercus_spectrogram_725ss2.jpg":   0.54,
    "Gryllus_pennsylvanicus_spectrogram_489ss2.gif": 0.54,
    "Gryllus_planeta_spectrogram_729ss3.jpg":        0.54,
    "Gryllus_veletis_spectrogram_488ss2.gif":        0.38,
    "Gryllus_vulcanus_spectrogram_738ss2.jpg":       0.62,
}

PLOT_TOP_BORDER = 6  # image rows to skip at the top (title bar, not signal)

## Helper Functions

In [ ]:
def gaussian_filter1d(signal, sigma):
    """Apply a 1-D Gaussian smoothing kernel to a 1-D array."""
    kernel_radius    = int(4 * sigma + 0.5)
    kernel_positions = np.arange(-kernel_radius, kernel_radius + 1, dtype=float)
    kernel           = np.exp(-0.5 * (kernel_positions / sigma) ** 2)
    kernel          /= kernel.sum()
    return np.convolve(signal, kernel, mode="same")


def silhouette(values, cluster_labels):
    """Mean silhouette score for a 1-D binary clustering."""
    total_score = 0.0
    for i, value in enumerate(values):
        same_cluster  = values[cluster_labels == cluster_labels[i]]
        other_cluster = values[cluster_labels != cluster_labels[i]]
        within_dist   = np.mean(np.abs(same_cluster  - value)) if len(same_cluster)  > 1 else 0.0
        between_dist  = np.mean(np.abs(other_cluster - value)) if len(other_cluster) > 0 else 0.0
        denom         = max(within_dist, between_dist)
        total_score  += (between_dist - within_dist) / denom if denom > 0 else 0.0
    return total_score / len(values)


def kmeans2(values):
    """Best 2-way split of a 1-D array by exhaustive search over split points.
    Returns (cluster_labels, cluster_centers); label 0 = short, 1 = long."""
    sorted_values    = np.sort(values)
    best_silhouette  = -2.0
    best_split_index = 1

    for i in range(1, len(sorted_values)):
        if i > 1 and sorted_values[i] == sorted_values[i - 1]:
            continue
        split_threshold  = (sorted_values[i - 1] + sorted_values[i]) / 2
        cluster_labels   = (values > split_threshold).astype(int)
        if len(set(cluster_labels)) < 2:
            continue
        split_silhouette = silhouette(values, cluster_labels)
        if split_silhouette > best_silhouette:
            best_silhouette  = split_silhouette
            best_split_index = i

    split_threshold = (sorted_values[best_split_index - 1] + sorted_values[best_split_index]) / 2
    cluster_labels  = (values > split_threshold).astype(int)
    cluster_centers = np.array([
        values[cluster_labels == 0].mean() if (cluster_labels == 0).any() else sorted_values[0],
        values[cluster_labels == 1].mean() if (cluster_labels == 1).any() else sorted_values[-1],
    ])
    return cluster_labels, cluster_centers


def choose_k(gap_durations):
    """Return 1 if gaps form one cluster, 2 if they split into two (requires silhouette > 0.3)."""
    gap_durations = np.asarray(gap_durations, dtype=float)
    if len(gap_durations) < 3 or len(np.unique(gap_durations)) < 2:
        return 1
    cluster_labels, _ = kmeans2(gap_durations)
    if len(set(cluster_labels)) < 2:
        return 1
    return 2 if silhouette(gap_durations, cluster_labels) > 0.3 else 1


def rle(signal_list):
    """Run-length encode a 1-D sequence."""
    run_list      = []
    current_value = signal_list[0]
    run_length    = 1
    for value in signal_list[1:]:
        if value == current_value:
            run_length += 1
        else:
            run_list.append((current_value, run_length))
            current_value = value
            run_length    = 1
    run_list.append((current_value, run_length))
    return run_list


def group_consecutive(active_indices, max_gap=3):
    """Group indices separated by <= max_gap into contiguous (start, end) spans (inclusive)."""
    if not len(active_indices):
        return []
    group_list  = []
    group_start = active_indices[0]
    last_index  = active_indices[0]
    for idx in active_indices[1:]:
        if idx - last_index <= max_gap:
            last_index = idx
        else:
            group_list.append((group_start, last_index))
            group_start = last_index = idx
    group_list.append((group_start, last_index))
    return group_list

## OCR and X-Axis

In [ ]:
def ocr_number_from_crop(label_image):
    """OCR a single numeric label from a small image crop; returns a float or None."""
    label_image    = label_image.convert("L")
    enlarged_image = label_image.resize(
        (label_image.width * 6, label_image.height * 6), Image.LANCZOS
    )
    enlarged_image = ImageOps.expand(enlarged_image, border=35, fill="white")
    gray_array     = np.array(enlarged_image)

    _, otsu_thresh  = cv2.threshold(gray_array, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive_thresh = cv2.adaptiveThreshold(
        gray_array, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11
    )

    ocr_configs = [
        r"--psm 7 -c tessedit_char_whitelist=0123456789.",
        r"--psm 8 -c tessedit_char_whitelist=0123456789.",
        r"--psm 13 -c tessedit_char_whitelist=0123456789.",
    ]
    for thresholded in [otsu_thresh, adaptive_thresh]:
        ocr_image = Image.fromarray(thresholded)
        for ocr_config in ocr_configs:
            text = (
                pytesseract.image_to_string(ocr_image, config=ocr_config).strip()
                .replace(" ", "").replace(",", ".").replace("O", "0")
                .replace("o", "0").replace("|", "1").replace("l", "1")
            )
            number_match = re.search(r"\d+(?:\.\d+)?", text)
            if number_match:
                return float(number_match.group())
    return None


def read_x_axis_seconds(spectrogram, manual_max_seconds=None):
    """
    Locate the x-axis line and return the time span it represents, in seconds.
    Tries OCR on the end labels first, falls back to fitting a line through
    several sampled interior labels; manual_max_seconds skips OCR entirely.

    Returns (max_seconds, plot_left, plot_right, axis_row).
    """
    gray_array            = np.array(spectrogram.convert("L"))
    img_height, img_width = gray_array.shape

    search_top    = int(img_height * 0.45)
    search_bottom = max(search_top + 1, img_height - 5)
    best_axis     = None

    for y in range(search_top, search_bottom):
        dark_pixels = np.where(gray_array[y, :] < 190)[0]
        for run_start, run_end in group_consecutive(dark_pixels, max_gap=3):
            run_width = run_end - run_start + 1
            if run_width > img_width * 0.40 and (best_axis is None or run_width > best_axis[0]):
                best_axis = (run_width, y, run_start, run_end)

    if best_axis is None:
        raise ValueError("No x-axis found in image")

    _, axis_row, plot_left, plot_right = best_axis
    plot_left, plot_right = int(plot_left), int(plot_right)

    if manual_max_seconds is not None:
        return manual_max_seconds, plot_left, plot_right, axis_row

    label_region_top    = min(img_height - 1, axis_row + 2)
    label_region_bottom = img_height
    detected_labels     = []

    for sample_x, crop_left, crop_right in [
        (plot_left,  max(0, plot_left  - 25), min(img_width, plot_left  + 90)),
        (plot_right, max(0, plot_right - 100), min(img_width, plot_right + 35)),
    ]:
        ocr_value = ocr_number_from_crop(
            spectrogram.crop((crop_left, label_region_top, crop_right, label_region_bottom))
        )
        if ocr_value is not None:
            detected_labels.append((sample_x, ocr_value))

    if len(detected_labels) >= 2:
        left_x,  left_val  = min(detected_labels, key=lambda t: t[0])
        right_x, right_val = max(detected_labels, key=lambda t: t[0])
        if right_val > left_val:
            return float(right_val), plot_left, plot_right, axis_row

    for frac in [0.25, 0.50, 0.75]:
        sample_x  = int(round(plot_left + frac * (plot_right - plot_left)))
        ocr_value = ocr_number_from_crop(
            spectrogram.crop((
                max(0, sample_x - 75), label_region_top,
                min(img_width, sample_x + 75), label_region_bottom,
            ))
        )
        if ocr_value is not None:
            detected_labels.append((sample_x, ocr_value))

    unique_labels = sorted(dict(detected_labels).items())
    if len(unique_labels) >= 2:
        label_x_positions = np.array([x for x, v in unique_labels])
        label_values      = np.array([v for x, v in unique_labels])
        slope, intercept  = np.polyfit(label_x_positions, label_values, 1)
        max_seconds       = float(slope * plot_right + intercept)
        right_x, right_val = max(unique_labels, key=lambda t: t[0])
        if abs(right_x - plot_right) < 35:
            max_seconds = float(right_val)
    else:
        raise ValueError(f"OCR failed--too few labels detected: {unique_labels}")

    return max_seconds, plot_left, plot_right, axis_row

## Signal Analysis

In [ ]:
def clean_signal_runs(signal_list, pixel_time, fill_gap_size=0):
    """Drop on-runs shorter than 3 ms. With fill_gap_size > 0, also fill
    off-gaps up to that many pixels wide between two on-runs (cricket keeps
    this at 0--real inter-element gaps can be a single pixel)."""
    signal_list   = np.asarray(signal_list, dtype=np.uint8)
    min_on_pixels = max(1, round(0.003 / pixel_time))

    cleaned = []
    for value, run_length in rle(signal_list):
        if value == 1 and run_length < min_on_pixels:
            cleaned.extend([0] * run_length)
        else:
            cleaned.extend([int(value)] * run_length)
    cleaned = np.array(cleaned, dtype=np.uint8)

    if fill_gap_size > 0 and len(cleaned) > 2:
        result_list = []
        run_list    = rle(cleaned)
        for i, (value, run_length) in enumerate(run_list):
            surrounded_by_signal = (
                i > 0 and i < len(run_list) - 1
                and run_list[i - 1][0] == 1 and run_list[i + 1][0] == 1
            )
            if value == 0 and surrounded_by_signal and run_length <= fill_gap_size:
                result_list.extend([1] * run_length)
            else:
                result_list.extend([int(value)] * run_length)
        cleaned = np.array(result_list, dtype=np.uint8)

    return cleaned


def locate_signal_band(ink, pad=5):
    """Row range of the dominant frequency band (cricket calls are narrowband).
    Returns (band_top_row, band_bottom_row)."""
    if ink.size == 0:
        raise ValueError("Empty ink array")

    row_pct95           = np.percentile(ink, 95, axis=1)
    row_pct99           = np.percentile(ink, 99, axis=1)
    row_signal_strength = gaussian_filter1d(0.35 * row_pct95 + 0.65 * row_pct99, sigma=1)

    row_floor = np.percentile(row_signal_strength, 10)
    row_peak  = np.percentile(row_signal_strength, 99)
    if row_peak <= row_floor:
        return 0, ink.shape[0]

    normalized_rows = np.clip(
        (row_signal_strength - row_floor) / (row_peak - row_floor), 0, 1
    )

    for threshold in [0.20, 0.14, 0.10, 0.06, 0.03]:
        active_rows      = np.where(normalized_rows >= threshold)[0]
        if not len(active_rows):
            continue
        candidate_groups = group_consecutive(active_rows, max_gap=1)
        group_start, group_end = max(
            candidate_groups,
            key=lambda ab: np.mean(normalized_rows[ab[0]:ab[1] + 1]) / (ab[1] - ab[0] + 1) ** 0.5
        )
        return max(0, group_start - pad), min(ink.shape[0], group_end + pad + 1)

    return 0, ink.shape[0]


def detect_signal_list_adaptive(ink_array, pixel_time, preferred_threshold=None):
    """Threshold the ink array into a binary on/off signal via a descending
    hysteresis-pair cascade, scored by threshold height minus a noise penalty.
    preferred_threshold pins the pick near a given value instead (for faint
    first elements the scorer misses). Returns (signal_list, normalized_signal)."""
    col_pct85           = np.percentile(ink_array, 85, axis=0)
    col_pct95           = np.percentile(ink_array, 95, axis=0)
    col_pct99           = np.percentile(ink_array, 99, axis=0)
    col_signal_strength = gaussian_filter1d(
        0.20 * col_pct85 + 0.35 * col_pct95 + 0.45 * col_pct99,
        sigma=0.20
    )

    signal_floor      = np.percentile(col_signal_strength,  5)
    signal_peak       = np.percentile(col_signal_strength, 99.5)
    if signal_peak <= signal_floor:
        raise ValueError("No contrast in ink array")
    normalized_signal = np.clip(
        (col_signal_strength - signal_floor) / (signal_peak - signal_floor), 0, 1
    )

    min_on_pixels        = max(1, round(0.003 / pixel_time))
    candidate_thresholds = []

    for high_threshold in [0.78, 0.70, 0.62, 0.54, 0.46, 0.38, 0.30, 0.22, 0.16, 0.10]:
        low_threshold = max(0.06, high_threshold * 0.45)
        high_mask     = normalized_signal >= high_threshold
        low_mask      = normalized_signal >= low_threshold

        signal_list = np.zeros_like(low_mask, dtype=np.uint8)
        column_pos  = 0
        for value, run_length in rle(low_mask.astype(np.uint8)):
            run_start, run_end = column_pos, column_pos + run_length
            if value == 1 and run_length >= min_on_pixels and np.any(high_mask[run_start:run_end]):
                signal_list[run_start:run_end] = 1
            column_pos = run_end

        signal_list     = clean_signal_runs(signal_list, pixel_time, fill_gap_size=0)
        active_fraction = float(signal_list.mean())
        if active_fraction <= 0 or active_fraction >= 0.88:
            continue

        on_run_lengths   = [run_length for value, run_length in rle(signal_list) if value == 1]
        if not on_run_lengths:
            continue

        median_on_length  = float(np.median(on_run_lengths))
        tiny_run_fraction = sum(l <= 1 for l in on_run_lengths) / len(on_run_lengths)
        score = high_threshold - 0.15 * tiny_run_fraction - 0.002 * median_on_length
        candidate_thresholds.append((score, high_threshold, signal_list, normalized_signal))

    if not candidate_thresholds:
        raise ValueError("No valid signal detected at any threshold")

    if preferred_threshold is not None:
        candidate_thresholds.sort(key=lambda t: abs(t[1] - preferred_threshold))
    else:
        candidate_thresholds.sort(key=lambda t: t[0], reverse=True)
    return candidate_thresholds[0][2], candidate_thresholds[0][3]

## Interval Classification

In [ ]:
def classify_intervals(time_bucket_list, leading_silence, trailing_silence,
                       element_length, pixel_time):
    """Classify off-run durations as inter-element vs. inter-burst by clustering
    them into two groups and checking guards (gap ratio, separation, position,
    pixel width) before trusting the split. Returns
    (inter_element_interval, inter_burst_interval, elements_per_burst); a trill
    gives (mean gap, 0, 1)."""
    off_durations = np.array(
        [duration for state, duration in time_bucket_list if state == "Off"],
        dtype=float,
    )

    if len(off_durations) == 0:
        edge_gaps = [x for x in [leading_silence, trailing_silence] if x > 0]
        return 0.0, float(np.mean(edge_gaps)) if edge_gaps else 0.0, 1

    if len(off_durations) < 4 or len(np.unique(np.round(off_durations, 6))) < 2:
        return float(np.mean(off_durations)), 0.0, 1

    if choose_k(off_durations) != 2:
        return float(np.mean(off_durations)), 0.0, 1

    cluster_labels, cluster_centers = kmeans2(off_durations)
    short_gap_durations = off_durations[cluster_labels == 0]
    long_gap_durations  = off_durations[cluster_labels == 1]
    inter_element_mean  = float(np.mean(short_gap_durations))
    inter_burst_mean    = float(np.mean(long_gap_durations))

    if inter_element_mean <= 0:
        return float(np.mean(off_durations)), 0.0, 1

    burst_gap_ratio    = inter_burst_mean / inter_element_mean
    long_gap_fraction  = len(long_gap_durations) / len(off_durations)
    n_long_gaps        = len(long_gap_durations)
    inter_burst_pixels = inter_burst_mean / pixel_time

    if n_long_gaps == 1:
        single_long_gap_index = int(np.where(cluster_labels == 1)[0][0])
        single_gap_position   = single_long_gap_index / max(1, len(off_durations) - 1)
    else:
        single_gap_position = 0.5

    is_burst = (
        burst_gap_ratio    >= 4.0
        and inter_burst_mean   >= 1.2 * element_length
        and long_gap_fraction  <= 0.70
        and inter_burst_pixels >= 5
        and (n_long_gaps >= 2 or (inter_burst_pixels >= 200 and single_gap_position >= 0.25))
    )

    # 3-cluster fallback: if only one gap looked long (often just leading silence),
    # re-cluster the "short" gaps on their own to check for a second, smaller-scale
    # burst split hiding underneath.
    if not is_burst and n_long_gaps == 1 and len(short_gap_durations) >= 6:
        if choose_k(short_gap_durations) == 2:
            secondary_labels, _ = kmeans2(short_gap_durations)
            secondary_silhouette = silhouette(short_gap_durations, secondary_labels)
            secondary_short = short_gap_durations[secondary_labels == 0]
            secondary_long  = short_gap_durations[secondary_labels == 1]
            if (secondary_silhouette > 0.5 and len(secondary_long) >= 2
                    and float(np.mean(secondary_long)) >= 4.0 * float(np.mean(secondary_short))
                    and float(np.mean(secondary_long)) / pixel_time >= 5):
                inter_element_mean = float(np.mean(secondary_short))
                inter_burst_mean   = float(np.mean(secondary_long))
                burst_gap_ratio    = inter_burst_mean / inter_element_mean
                long_gap_fraction  = len(secondary_long) / len(short_gap_durations)
                inter_burst_pixels = inter_burst_mean / pixel_time
                n_long_gaps        = len(secondary_long)
                is_burst = (
                    burst_gap_ratio    >= 4.0
                    and inter_burst_mean   >= 1.2 * element_length
                    and long_gap_fraction  <= 0.70
                    and inter_burst_pixels >= 5
                    and n_long_gaps        >= 2
                )

    if not is_burst:
        return float(np.mean(off_durations)), 0.0, 1

    burst_boundary_time     = (inter_element_mean + inter_burst_mean) / 2
    elements_per_burst_list = []
    current_count           = 0
    for state, duration in time_bucket_list:
        if state == "On":
            current_count += 1
        elif duration > burst_boundary_time:
            if current_count > 0:
                elements_per_burst_list.append(current_count)
            current_count = 0
    if current_count > 0:
        elements_per_burst_list.append(current_count)

    if not elements_per_burst_list:
        return float(np.mean(off_durations)), 0.0, 1
    med_epb = statistics.median(elements_per_burst_list)
    if med_epb < 1.5:
        return float(np.mean(off_durations)), 0.0, 1

    # Trim edge bursts that were likely clipped by the recording's start/end.
    if len(elements_per_burst_list) >= 4:
        overall_med    = statistics.median(elements_per_burst_list)
        trim_threshold = overall_med / 3.0
        trimmed        = list(elements_per_burst_list)
        if trimmed[0] < trim_threshold:
            trimmed = trimmed[1:]
        if len(trimmed) > 1 and trimmed[-1] < trim_threshold:
            trimmed = trimmed[:-1]
        if trimmed:
            med_epb = statistics.median(trimmed)
            if med_epb < 1.5:
                return float(np.mean(off_durations)), 0.0, 1

    return inter_element_mean, inter_burst_mean, max(1, round(med_epb))

## Processing Function

In [ ]:
def cricket_process(file_path):
    """Extract the four acoustic parameters from one cricket spectrogram:
    OCR x-axis → locate frequency band → threshold ink → trim silence → classify gaps."""
    # Return hand-coded values for species where image resolution is too low.
    manual = MANUAL_RESULTS.get(Path(file_path).name)
    if manual:
        return {k: round(v, 4) if isinstance(v, float) else v
                for k, v in manual.items()}

    spectrogram = Image.open(file_path).convert("L")
    spec_array  = np.array(spectrogram)

    manual_max_seconds = MANUAL_MAX_SECONDS.get(Path(file_path).name)
    max_seconds, plot_left, plot_right, axis_row = read_x_axis_seconds(
        spectrogram, manual_max_seconds
    )
    axis_width = plot_right - plot_left
    if axis_width <= 0:
        raise ValueError(f"Invalid x-axis width: {axis_width} px")
    pixel_time = max_seconds / axis_width

    plot_bottom = max(PLOT_TOP_BORDER + 1, axis_row - 2)
    plot_area   = spec_array[PLOT_TOP_BORDER:plot_bottom, plot_left:plot_right]
    if plot_area.size == 0:
        raise ValueError("Empty plot area")

    background_level = np.percentile(plot_area, 95)
    ink              = np.clip(background_level - plot_area.astype(float), 0, None)
    ink_peak         = np.percentile(ink, 99)
    if ink_peak > 0:
        ink /= ink_peak

    # Cricket calls are narrowband--restrict to dominant frequency band.
    # pad=5 keeps a 5-row margin; narrowband scoring avoids fat noise bands.
    band_top, band_bottom = locate_signal_band(ink, pad=5)
    band_ink              = ink[band_top:band_bottom, :]
    band_ink_peak = np.percentile(band_ink, 99)
    if band_ink_peak > 0:
        band_ink /= band_ink_peak  # re-normalise within band

    faint_threshold = FAINT_ELEMENT_THRESHOLD.get(Path(file_path).name)
    raw_signal_list, _ = detect_signal_list_adaptive(
        band_ink, pixel_time, preferred_threshold=faint_threshold
    )

    signal_columns = np.where(raw_signal_list == 1)[0]
    if not len(signal_columns):
        raise ValueError("No signal columns detected")

    trim_start = max(0, signal_columns[0]  - 2)
    trim_end   = min(len(raw_signal_list), signal_columns[-1] + 3)
    leading_silence  = trim_start                         * pixel_time
    trailing_silence = (len(raw_signal_list) - trim_end) * pixel_time

    signal_list = clean_signal_runs(
        raw_signal_list[trim_start:trim_end], pixel_time, fill_gap_size=0
    )
    if not len(signal_list) or signal_list.mean() == 0:
        raise ValueError("Empty signal after cleanup")

    time_buckets = [
        ("On" if value == 1 else "Off", run_length * pixel_time)
        for value, run_length in rle(signal_list)
    ]
    on_durations = [duration for state, duration in time_buckets if state == "On"]
    if not on_durations:
        raise ValueError("No on-pulses detected")
    element_length = float(np.mean(on_durations))

    # Fallback for long-element species where a few short on-runs are real elements
    # embedded in a mostly-long pattern (e.g. Anurogryllus with MANUAL_MAX_SECONDS).
    if element_length > 0.5 and len(on_durations) < 8:
        short_on = [d for d in on_durations if d < 0.3]
        if len(short_on) >= 2:
            element_length = float(np.mean(short_on))
            short_time_buckets = [
                (state, dur) for state, dur in time_buckets
                if state == "Off" or dur < 0.3
            ]
            short_off_durs = [
                dur for state, dur in short_time_buckets
                if state == "Off" and dur < 2.0
            ]
            if short_off_durs:
                return {
                    "element_length":         round(element_length, 4),
                    "inter_element_interval": round(float(np.mean(short_off_durs)), 4),
                    "inter_burst_interval":   0.0,
                    "elements_per_burst":     1,
                }

    inter_element_interval, inter_burst_interval, elements_per_burst = classify_intervals(
        time_buckets, leading_silence, trailing_silence, element_length, pixel_time
    )

    return {
        "element_length":         round(element_length,         4),
        "inter_element_interval": round(inter_element_interval, 4),
        "inter_burst_interval":   round(inter_burst_interval,   4),
        "elements_per_burst":     elements_per_burst,
    }

## Run Pipeline on All Spectrograms

Runs every spectrogram in `Crickets/` through the pipeline and saves the raw results to `cricket_results.csv`.

In [ ]:
spec_files = sorted(
    p for p in CRICKETS_DIR.rglob("*spectrogram*")
    if "checkpoint" not in str(p)
    and p.suffix.lower() in {".gif", ".jpg", ".jpeg", ".png"}
)

rows = []
for file_path in spec_files:
    species_folder = file_path.parent.name
    if species_folder in {"Hapithus_melodius",   # excluded: accelerating chirps
                            "Gryllus_cayensis",    # excluded: courtship song
                            "Gryllus_ovisopis"}:   # excluded: two males fighting
        continue
    try:
        result = cricket_process(file_path)
        status = "ok"; error = ""
    except Exception as e:
        result = {}; status = "error"; error = str(e)
    rows.append({"species": species_folder, "file": file_path.name,
                 "status": status, "error": error, **result})

output_csv = CRICKETS_DIR / "cricket_results.csv"
import csv
csv_columns = ["species", "file", "status", "error",
               "element_length", "inter_element_interval",
               "inter_burst_interval", "elements_per_burst"]
with open(output_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_columns, extrasaction="ignore")
    writer.writeheader(); writer.writerows(rows)

successful_rows = [r for r in rows if r["status"] == "ok"]
error_rows      = [r for r in rows if r["status"] == "error"]
print(f"Processed {len(rows)} spectrograms: {len(successful_rows)} ok, {len(error_rows)} errors")
for r in error_rows:
    print(f"  ERROR {r['species']}: {r['error']}")

## Results

In [ ]:
df    = pd.read_csv(output_csv)
df_ok = df[df["status"] == "ok"].copy()
print(f"{len(df_ok)} spectrograms processed successfully")
df_ok[["species", "element_length", "inter_element_interval",
       "inter_burst_interval", "elements_per_burst"]]

## Merge with Webscraping Data

Joins the pipeline output to `cricket_df` (from `Webscraping.ipynb`) on the
spectrogram's `File_ID`, e.g. `548ss`. **Run `Webscraping.ipynb` at least once
first.**

In [ ]:
import os

# Requires %store cricket_df to have been run in Webscraping.ipynb
%store -r cricket_df


def spec_id_from_url(url):
    """'https://orthsoc.org/sina/548ss.gif' -> '548ss'"""
    if pd.isna(url): return None
    return os.path.splitext(os.path.basename(str(url)))[0]


def spec_id_from_filename(filename):
    """'Acheta_domesticus_spectrogram_548ss.gif' -> '548ss'"""
    match = re.search(r'_spectrogram_(.+)\.[^.]+$', str(filename))
    return match.group(1) if match else None


webscraping_data = cricket_df[[
    'Species', 'Temperature (°C)', 'Location',
    'Description of Whole Audio File', 'Map', 'Spectrogram',
]].copy()
webscraping_data['_spec_id'] = webscraping_data['Spectrogram'].apply(spec_id_from_url)

processing_results = df_ok[[
    'file', 'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
]].copy()
processing_results['_spec_id'] = processing_results['file'].apply(spec_id_from_filename)

merged = processing_results.merge(
    webscraping_data.drop(columns='Spectrogram'), on='_spec_id', how='left'
)

merged[['Genus', 'Species']] = merged['Species'].str.split(' ', n=1, expand=True)

# Column names below match Display_Function.ipynb exactly -- do not rename.
# File_ID is the spectrogram identifier from the SINA URL (e.g. '548ss').
cricket_final = merged[[
    'Genus', 'Species',
    'Temperature (°C)', 'Location',
    'Description of Whole Audio File', 'Map',
    '_spec_id',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
]].rename(columns={
    'Description of Whole Audio File': 'Description',
    'Temperature (°C)':           'Temperature',
    '_spec_id':                        'File_ID',
    'element_length':                  'Element_Length',
    'inter_element_interval':          'Inter-Element_Interval',
    'inter_burst_interval':            'Inter-Burst_Interval',
    'elements_per_burst':              'Elements_Per_Burst',
})

print(f'{len(cricket_final)} rows | {cricket_final["Genus"].nunique()} genera'
      f' | {cricket_final["Species"].nunique()} species')
cricket_final


In [ ]:
%store cricket_final

In [ ]:
cricket_final.to_csv(Path.home() / "Discrete_Signals" / "cricket.csv", index=False)